In [0]:
sensor_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/default/industrial_iot_data/sensor_readings.csv")
)


In [0]:
sensor_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_sensor_readings")

In [0]:
# Verify Bronze sensor data

display(
    spark.table("workspace.default.bronze_sensor_readings")
)

MachineID,ReadingTime,Temperature,Pressure,Flow,Vibration
P101,2026-08-01T00:00:00.000Z,70.71,86.83,142.01,2.68
P101,2026-08-01T01:00:00.000Z,69.0,90.77,129.58,3.09
P101,2026-08-01T02:00:00.000Z,74.89,100.17,158.55,2.47
P101,2026-08-01T03:00:00.000Z,68.41,85.23,131.09,3.7
P101,2026-08-01T04:00:00.000Z,71.2,84.92,105.16,3.32
P101,2026-08-01T05:00:00.000Z,77.2,101.52,108.02,2.99
P101,2026-08-01T06:00:00.000Z,72.82,97.62,135.69,2.98
P101,2026-08-01T07:00:00.000Z,76.75,95.81,125.2,3.31
P101,2026-08-01T08:00:00.000Z,61.8,91.16,111.53,2.29
P101,2026-08-01T09:00:00.000Z,70.07,112.94,104.42,3.57


In [0]:
equipment_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/default/industrial_iot_data/equipment_master.csv")
)

display(equipment_df)

MachineID,EquipmentName,EquipmentType,Plant,Manufacturer,CommissionYear
P101,Pump-101,Pump,Plant-A,Siemens,2022
P102,Pump-102,Pump,Plant-B,Schneider,2019
P103,Pump-103,Pump,Plant-C,ABB,2023
P104,Pump-104,Pump,Plant-A,Siemens,2022
P105,Pump-105,Pump,Plant-B,Siemens,2016
T101,Tank-101,Tank,Plant-A,Schneider,2024
T102,Tank-102,Tank,Plant-B,Schneider,2022
T103,Tank-103,Tank,Plant-C,Schneider,2023
T104,Tank-104,Tank,Plant-A,Schneider,2017
T105,Tank-105,Tank,Plant-B,GE,2020


In [0]:
equipment_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_equipment_master")

In [0]:
maintenance_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/default/industrial_iot_data/maintenance_records.csv")
)

display(maintenance_df)

MaintenanceID,MachineID,MaintenanceDate,MaintenanceType,DowntimeHours,Cost,TechnicianStatus
MT-P101-01,P101,2026-07-19,Inspection,6.6,16055,Follow-up Required
MT-P101-02,P101,2026-08-17,Preventive Maintenance,2.8,14947,Completed
MT-P101-03,P101,2026-07-02,Inspection,7.6,11461,Follow-up Required
MT-P102-01,P102,2026-07-05,Seal Replacement,1.3,9848,Completed
MT-P102-02,P102,2026-06-16,Lubrication,2.0,14960,Completed
MT-P102-03,P102,2026-06-10,Bearing Check,4.9,13834,Follow-up Required
MT-P103-01,P103,2026-08-17,Inspection,3.0,8656,Follow-up Required
MT-P103-02,P103,2026-07-11,Bearing Check,5.9,23979,Completed
MT-P103-03,P103,2026-08-02,Preventive Maintenance,4.9,22688,Completed
MT-P104-01,P104,2026-07-11,Preventive Maintenance,4.4,2491,Completed


In [0]:
maintenance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_maintenance_records")

In [0]:
%sql

SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,bronze_equipment_master,false
default,bronze_maintenance_records,false
default,bronze_sensor_readings,false
default,gold_daily_sensor_summary,false
default,gold_machine_kpi,false
default,silver_equipment_master,false
default,silver_maintenance_records,false
default,silver_sensor_readings,false


In [0]:
print("Sensor:", spark.table(
    "workspace.default.bronze_sensor_readings"
).count())

print("Equipment:", spark.table(
    "workspace.default.bronze_equipment_master"
).count())

print("Maintenance:", spark.table(
    "workspace.default.bronze_maintenance_records"
).count())

Sensor: 6755
Equipment: 20
Maintenance: 60


In [0]:
silver_sensor_df = spark.table(
    "workspace.default.bronze_sensor_readings"
)

In [0]:
# Reusable transformation function for Bronze -> Silver sensor data

from pyspark.sql import Window
from pyspark.sql.functions import col, avg, when, to_timestamp

def clean_sensor_data(df):

    # Remove duplicate sensor readings
    df = df.dropDuplicates(
        ["MachineID", "ReadingTime"]
    )

    # Convert ReadingTime to timestamp
    df = df.withColumn(
        "ReadingTime",
        to_timestamp("ReadingTime")
    )

    # Fill missing Temperature and Pressure
    # using the average value for the same machine
    machine_window = Window.partitionBy("MachineID")

    df = (
        df
        .withColumn(
            "Temperature",
            when(
                col("Temperature").isNull(),
                avg("Temperature").over(machine_window)
            ).otherwise(col("Temperature"))
        )
        .withColumn(
            "Pressure",
            when(
                col("Pressure").isNull(),
                avg("Pressure").over(machine_window)
            ).otherwise(col("Pressure"))
        )
    )

    # Remove invalid negative sensor values
    df = df.filter(
        (col("Temperature") >= 0) &
        (col("Pressure") >= 0) &
        (col("Flow") >= 0) &
        (col("Vibration") >= 0)
    )

    # Temperature status
    df = df.withColumn(
        "TemperatureStatus",
        when(col("Temperature") >= 100, "Critical")
        .when(col("Temperature") >= 90, "Warning")
        .otherwise("Normal")
    )

    # Pressure status
    df = df.withColumn(
        "PressureStatus",
        when(col("Pressure") >= 140, "Critical")
        .when(col("Pressure") >= 110, "Warning")
        .otherwise("Normal")
    )

    # Overall machine status
    df = df.withColumn(
        "MachineStatus",
        when(
            (col("Temperature") >= 100) |
            (col("Pressure") >= 140) |
            (col("Vibration") >= 7),
            "Critical"
        )
        .when(
            (col("Temperature") >= 90) |
            (col("Pressure") >= 110) |
            (col("Vibration") >= 5),
            "Warning"
        )
        .otherwise("Normal")
    )

    return df

In [0]:
# Apply all Silver cleaning and validation rules

silver_sensor_df = clean_sensor_data(
    silver_sensor_df
)

display(silver_sensor_df)

MachineID,ReadingTime,Temperature,Pressure,Flow,Vibration,TemperatureStatus,PressureStatus,MachineStatus
C101,2026-08-01T00:00:00.000Z,92.3,138.08,178.55,4.19,Warning,Warning,Warning
C101,2026-08-01T01:00:00.000Z,82.02499999999993,133.44,139.61,4.1,Normal,Warning,Warning
C101,2026-08-01T02:00:00.000Z,80.77,107.47,152.29,3.65,Normal,Normal,Normal
C101,2026-08-01T03:00:00.000Z,80.3,129.37,172.91,3.32,Normal,Warning,Warning
C101,2026-08-01T04:00:00.000Z,79.64,115.82,135.06,3.52,Normal,Warning,Warning
C101,2026-08-01T05:00:00.000Z,80.62,132.03,155.77,3.15,Normal,Warning,Warning
C101,2026-08-01T06:00:00.000Z,91.06,136.33,133.78,4.98,Warning,Warning,Warning
C101,2026-08-01T07:00:00.000Z,84.29,123.12,156.45,3.98,Normal,Warning,Warning
C101,2026-08-01T08:00:00.000Z,85.82,127.25,143.93,2.38,Normal,Warning,Warning
C101,2026-08-01T09:00:00.000Z,85.69,120.52,164.0,4.39,Normal,Warning,Warning


In [0]:
# Reusable function to MERGE incremental records into Silver

from delta.tables import DeltaTable

def merge_into_silver(new_silver_df):

    silver_table = DeltaTable.forName(
        spark,
        "workspace.default.silver_sensor_readings"
    )

    (
        silver_table.alias("target")
        .merge(
            new_silver_df.alias("source"),
            """
            target.MachineID = source.MachineID
            AND target.ReadingTime = source.ReadingTime
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:


silver_sensor_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_sensor_readings")

In [0]:
display(
    spark.table("workspace.default.silver_sensor_readings")
)

MachineID,ReadingTime,Temperature,Pressure,Flow,Vibration,TemperatureStatus,PressureStatus,MachineStatus
C101,2026-08-02T00:00:00.000Z,82.21,135.77,114.54,3.81,Normal,Warning,Warning
C101,2026-08-02T18:00:00.000Z,81.86,128.43,136.72,4.91,Normal,Warning,Warning
C101,2026-08-03T09:00:00.000Z,88.84,116.0,148.94,3.64,Normal,Warning,Warning
C101,2026-08-03T16:00:00.000Z,84.94,132.44,162.24,3.32,Normal,Warning,Warning
C101,2026-08-04T22:00:00.000Z,92.44,105.49,144.84,3.57,Warning,Normal,Warning
C101,2026-08-05T02:00:00.000Z,84.61,121.7,157.16,4.08,Normal,Warning,Warning
C101,2026-08-05T14:00:00.000Z,91.17,139.97,163.03,2.64,Warning,Warning,Warning
C101,2026-08-06T10:00:00.000Z,79.1,140.35,154.66,2.87,Normal,Critical,Critical
C101,2026-08-06T14:00:00.000Z,83.68,128.31,147.97,4.08,Normal,Warning,Warning
C101,2026-08-06T18:00:00.000Z,75.21,129.22,154.27,4.77,Normal,Warning,Warning


In [0]:
# Read raw equipment data from the Bronze Delta table

silver_equipment_df = spark.table(
    "workspace.default.bronze_equipment_master"
)

silver_equipment_df.show()

+---------+--------------+-------------+-------+------------+--------------+
|MachineID| EquipmentName|EquipmentType|  Plant|Manufacturer|CommissionYear|
+---------+--------------+-------------+-------+------------+--------------+
|     P101|      Pump-101|         Pump|Plant-A|     Siemens|          2022|
|     P102|      Pump-102|         Pump|Plant-B|   Schneider|          2019|
|     P103|      Pump-103|         Pump|Plant-C|         ABB|          2023|
|     P104|      Pump-104|         Pump|Plant-A|     Siemens|          2022|
|     P105|      Pump-105|         Pump|Plant-B|     Siemens|          2016|
|     T101|      Tank-101|         Tank|Plant-A|   Schneider|          2024|
|     T102|      Tank-102|         Tank|Plant-B|   Schneider|          2022|
|     T103|      Tank-103|         Tank|Plant-C|   Schneider|          2023|
|     T104|      Tank-104|         Tank|Plant-A|   Schneider|          2017|
|     T105|      Tank-105|         Tank|Plant-B|          GE|          2020|

In [0]:
# Remove duplicate equipment records based on MachineID
# Each MachineID should represent only one machine in the master table

silver_equipment_df = silver_equipment_df.dropDuplicates(
    ["MachineID"]
)

In [0]:
# Remove records where important equipment information is missing
# MachineID, EquipmentType and Plant are required for further processing

silver_equipment_df = silver_equipment_df.dropna(
    subset=["MachineID", "EquipmentType", "Plant"]
)

In [0]:
# Store the cleaned equipment data as a Silver Delta table

silver_equipment_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_equipment_master")

In [0]:
# Verify the cleaned Silver equipment data

silver_equipment_df.show()

print("Silver Equipment Count:", silver_equipment_df.count())

+---------+--------------+-------------+-------+------------+--------------+
|MachineID| EquipmentName|EquipmentType|  Plant|Manufacturer|CommissionYear|
+---------+--------------+-------------+-------+------------+--------------+
|     P101|      Pump-101|         Pump|Plant-A|     Siemens|          2022|
|     P102|      Pump-102|         Pump|Plant-B|   Schneider|          2019|
|     P103|      Pump-103|         Pump|Plant-C|         ABB|          2023|
|     P104|      Pump-104|         Pump|Plant-A|     Siemens|          2022|
|     P105|      Pump-105|         Pump|Plant-B|     Siemens|          2016|
|     T101|      Tank-101|         Tank|Plant-A|   Schneider|          2024|
|     T102|      Tank-102|         Tank|Plant-B|   Schneider|          2022|
|     T103|      Tank-103|         Tank|Plant-C|   Schneider|          2023|
|     T104|      Tank-104|         Tank|Plant-A|   Schneider|          2017|
|     T105|      Tank-105|         Tank|Plant-B|          GE|          2020|

In [0]:
# Read maintenance records from the Bronze Delta table

silver_maintenance_df = spark.table(
    "workspace.default.bronze_maintenance_records"
)

silver_maintenance_df.show()

+-------------+---------+---------------+--------------------+-------------+-----+------------------+
|MaintenanceID|MachineID|MaintenanceDate|     MaintenanceType|DowntimeHours| Cost|  TechnicianStatus|
+-------------+---------+---------------+--------------------+-------------+-----+------------------+
|   MT-P101-01|     P101|     2026-07-19|          Inspection|          6.6|16055|Follow-up Required|
|   MT-P101-02|     P101|     2026-08-17|Preventive Mainte...|          2.8|14947|         Completed|
|   MT-P101-03|     P101|     2026-07-02|          Inspection|          7.6|11461|Follow-up Required|
|   MT-P102-01|     P102|     2026-07-05|    Seal Replacement|          1.3| 9848|         Completed|
|   MT-P102-02|     P102|     2026-06-16|         Lubrication|          2.0|14960|         Completed|
|   MT-P102-03|     P102|     2026-06-10|       Bearing Check|          4.9|13834|Follow-up Required|
|   MT-P103-01|     P103|     2026-08-17|          Inspection|          3.0| 8656|

In [0]:
# Remove duplicate maintenance records using MaintenanceID

silver_maintenance_df = silver_maintenance_df.dropDuplicates(
    ["MaintenanceID"]
)

In [0]:
# Remove records that do not have a valid machine reference

silver_maintenance_df = silver_maintenance_df.dropna(
    subset=["MachineID", "MaintenanceDate"]
)

In [0]:
from pyspark.sql.functions import to_date

# Convert MaintenanceDate from string to proper DateType

silver_maintenance_df = silver_maintenance_df.withColumn(
    "MaintenanceDate",
    to_date("MaintenanceDate")
)

In [0]:
# Save the cleaned maintenance data as a Silver Delta table

silver_maintenance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_maintenance_records")

In [0]:
# Display the cleaned Silver maintenance dataset
display(silver_maintenance_df)

# Validate the total number of records
print(
    "Silver Maintenance Count:",
    silver_maintenance_df.count()
)

MaintenanceID,MachineID,MaintenanceDate,MaintenanceType,DowntimeHours,Cost,TechnicianStatus
MT-P101-01,P101,2026-07-19,Inspection,6.6,16055,Follow-up Required
MT-P101-02,P101,2026-08-17,Preventive Maintenance,2.8,14947,Completed
MT-P101-03,P101,2026-07-02,Inspection,7.6,11461,Follow-up Required
MT-P102-01,P102,2026-07-05,Seal Replacement,1.3,9848,Completed
MT-P102-02,P102,2026-06-16,Lubrication,2.0,14960,Completed
MT-P102-03,P102,2026-06-10,Bearing Check,4.9,13834,Follow-up Required
MT-P103-01,P103,2026-08-17,Inspection,3.0,8656,Follow-up Required
MT-P103-02,P103,2026-07-11,Bearing Check,5.9,23979,Completed
MT-P103-03,P103,2026-08-02,Preventive Maintenance,4.9,22688,Completed
MT-P104-01,P104,2026-07-11,Preventive Maintenance,4.4,2491,Completed


Silver Maintenance Count: 60


## 🥇 Gold Layer - Machine KPI

Create a business-ready machine-level KPI dataset by aggregating
cleaned sensor data and combining it with equipment information.

In [0]:
from pyspark.sql.functions import to_date, avg, max, sum, when, col

# Add a ReadingDate column for daily aggregation

daily_df = silver_sensor_df.withColumn(
    "ReadingDate",
    to_date("ReadingTime")
)

# Create daily machine-level KPIs

gold_daily_df = daily_df.groupBy(
    "MachineID",
    "ReadingDate"
).agg(
    avg("Temperature").alias("AvgTemperature"),
    max("Temperature").alias("MaxTemperature"),
    avg("Pressure").alias("AvgPressure"),
    max("Vibration").alias("MaxVibration"),
    sum(
        when(col("MachineStatus") == "Critical", 1)
        .otherwise(0)
    ).alias("CriticalAlertCount")
)

display(gold_daily_df)

MachineID,ReadingDate,AvgTemperature,MaxTemperature,AvgPressure,MaxVibration,CriticalAlertCount
C101,2026-08-01,82.80937499999997,92.3,125.95583333333333,5.74,3
C101,2026-08-02,79.05125,92.96,127.60541666666661,5.54,3
C101,2026-08-03,81.44125000000001,94.96,126.53000000000002,5.39,3
C101,2026-08-04,83.79291666666667,95.42,128.90636352295405,5.86,3
C101,2026-08-05,83.59291666666668,95.34,130.01583333333335,5.24,4
C101,2026-08-06,82.075,98.57,123.24916666666662,5.61,2
C101,2026-08-07,79.35458333333334,93.47,122.80469685628742,4.77,1
C101,2026-08-08,77.81458333333335,90.62,126.73666666666668,5.57,3
C101,2026-08-09,82.03291666666668,94.2,124.545,6.15,4
C101,2026-08-10,83.54041666666664,108.72,122.72125,5.75,2


In [0]:
from pyspark.sql.functions import avg, max, sum, when

# Calculate machine-level sensor KPIs from the Silver sensor table

sensor_kpi_df = silver_sensor_df.groupBy("MachineID").agg(

    avg("Temperature").alias("AvgTemperature"),

    max("Temperature").alias("MaxTemperature"),

    avg("Pressure").alias("AvgPressure"),

    avg("Vibration").alias("AvgVibration"),

    sum(
        when(
            silver_sensor_df["TemperatureStatus"] == "Critical", 1
        ).otherwise(0)
    ).alias("CriticalAlerts")
)

display(sensor_kpi_df)

MachineID,AvgTemperature,MaxTemperature,AvgPressure,AvgVibration,CriticalAlerts
C101,82.02499999999995,108.72,125.48272455089825,4.045535714285717,1
C102,81.49498507462697,101.6,125.20714285714291,4.11532738095238,1
C103,82.44471471471468,116.5,127.1447462686566,4.060386904761906,5
C104,82.43102102102095,117.05,125.89588059701488,4.035059523809529,2
C105,82.7510479041916,108.93,127.11017910447758,4.151011904761908,5
M101,76.25699404761914,124.02,20.99473214285715,3.1961607142857127,2
M102,76.1532142857143,116.64,19.71928358208955,3.2464285714285706,1
M103,76.74489552238803,110.52,22.09499999999999,3.156666666666667,1
M104,76.4893113772455,123.34,20.80543543543543,3.2938988095238066,3
M105,76.24673652694608,95.24,22.553053892215566,3.2780059523809553,0


In [0]:
from pyspark.sql.functions import sum, count

# Calculate maintenance KPIs for each machine

maintenance_kpi_df = silver_maintenance_df.groupBy("MachineID").agg(
    sum("Cost").alias("TotalMaintenanceCost"),
    sum("DowntimeHours").alias("TotalDowntimeHours"),
    count("MaintenanceID").alias("MaintenanceCount")
)

display(maintenance_kpi_df)

MachineID,TotalMaintenanceCost,TotalDowntimeHours,MaintenanceCount
P101,42463,17.0,3
P102,38642,8.2,3
P103,55323,13.8,3
P104,27712,13.6,3
P105,37292,14.1,3
T101,51915,9.8,3
T102,55404,8.0,3
T103,47716,14.499999999999998,3
T104,6582,12.3,3
T105,14948,17.5,3


In [0]:
# Combine machine sensor KPIs with equipment master information

from pyspark.sql.functions import broadcast

# Broadcast the small equipment master table to avoid unnecessary shuffle
gold_machine_df = sensor_kpi_df.join(
    broadcast(silver_equipment_df),
    "MachineID",
    "left"
)
# Add maintenance KPIs

gold_machine_df = gold_machine_df.join(
    maintenance_kpi_df,
    "MachineID",
    "left"
)

display(gold_machine_df)

MachineID,AvgTemperature,MaxTemperature,AvgPressure,AvgVibration,CriticalAlerts,EquipmentName,EquipmentType,Plant,Manufacturer,CommissionYear,TotalMaintenanceCost,TotalDowntimeHours,MaintenanceCount
C101,82.02500000000002,108.72,125.48272455089821,4.045535714285717,1,Compressor-101,Compressor,Plant-A,Schneider,2019,59205,15.1,3
C102,81.49498507462692,101.6,125.20714285714298,4.11532738095238,1,Compressor-102,Compressor,Plant-B,Siemens,2024,43669,19.6,3
C103,82.4447147147147,116.5,127.14474626865665,4.0603869047619,5,Compressor-103,Compressor,Plant-C,GE,2021,34630,16.2,3
C104,82.43102102102108,117.05,125.89588059701494,4.035059523809529,2,Compressor-104,Compressor,Plant-A,ABB,2023,45355,14.2,3
C105,82.75104790419157,108.93,127.11017910447771,4.151011904761905,5,Compressor-105,Compressor,Plant-B,Schneider,2019,46207,12.4,3
M101,76.2569940476191,124.02,20.994732142857153,3.196160714285714,2,Motor-101,Motor,Plant-A,ABB,2018,46109,12.8,3
M102,76.15321428571431,116.64,19.719283582089545,3.2464285714285728,1,Motor-102,Motor,Plant-B,Siemens,2020,51266,11.0,3
M103,76.74489552238803,110.52,22.09500000000001,3.156666666666665,1,Motor-103,Motor,Plant-C,GE,2016,44722,11.3,3
M104,76.48931137724546,123.34,20.80543543543543,3.2938988095238075,3,Motor-104,Motor,Plant-A,GE,2023,27208,9.100000000000001,3
M105,76.24673652694612,95.24,22.55305389221559,3.278005952380955,0,Motor-105,Motor,Plant-B,ABB,2021,27184,8.0,3


In [0]:
# Replace missing maintenance metrics with zero

gold_machine_df = gold_machine_df.fillna({
    "TotalMaintenanceCost": 0,
    "TotalDowntimeHours": 0,
    "MaintenanceCount": 0
})

display(gold_machine_df)

MachineID,AvgTemperature,MaxTemperature,AvgPressure,AvgVibration,CriticalAlerts,EquipmentName,EquipmentType,Plant,Manufacturer,CommissionYear,TotalMaintenanceCost,TotalDowntimeHours,MaintenanceCount
C101,82.02500000000002,108.72,125.48272455089821,4.045535714285717,1,Compressor-101,Compressor,Plant-A,Schneider,2019,59205,15.1,3
C102,81.49498507462692,101.6,125.20714285714298,4.11532738095238,1,Compressor-102,Compressor,Plant-B,Siemens,2024,43669,19.6,3
C103,82.4447147147147,116.5,127.14474626865665,4.0603869047619,5,Compressor-103,Compressor,Plant-C,GE,2021,34630,16.2,3
C104,82.43102102102108,117.05,125.89588059701494,4.035059523809529,2,Compressor-104,Compressor,Plant-A,ABB,2023,45355,14.2,3
C105,82.75104790419157,108.93,127.11017910447771,4.151011904761905,5,Compressor-105,Compressor,Plant-B,Schneider,2019,46207,12.4,3
M101,76.2569940476191,124.02,20.994732142857153,3.196160714285714,2,Motor-101,Motor,Plant-A,ABB,2018,46109,12.8,3
M102,76.15321428571431,116.64,19.719283582089545,3.2464285714285728,1,Motor-102,Motor,Plant-B,Siemens,2020,51266,11.0,3
M103,76.74489552238803,110.52,22.09500000000001,3.156666666666665,1,Motor-103,Motor,Plant-C,GE,2016,44722,11.3,3
M104,76.48931137724546,123.34,20.80543543543543,3.2938988095238075,3,Motor-104,Motor,Plant-A,GE,2023,27208,9.100000000000001,3
M105,76.24673652694612,95.24,22.55305389221559,3.278005952380955,0,Motor-105,Motor,Plant-B,ABB,2021,27184,8.0,3


In [0]:
# Save the final machine-level Gold KPI table

gold_machine_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_machine_kpi")

In [0]:
# Save the daily summary as a Gold Delta table

gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_sensor_summary")

In [0]:
# Verify the final Gold table

display(
    spark.table("workspace.default.gold_machine_kpi")
)

MachineID,AvgTemperature,MaxTemperature,AvgPressure,AvgVibration,CriticalAlerts,EquipmentName,EquipmentType,Plant,Manufacturer,CommissionYear,TotalMaintenanceCost,TotalDowntimeHours,MaintenanceCount
C101,82.02500000000002,108.72,125.48272455089821,4.045535714285717,1,Compressor-101,Compressor,Plant-A,Schneider,2019,59205,15.1,3
C102,81.49498507462692,101.6,125.20714285714298,4.11532738095238,1,Compressor-102,Compressor,Plant-B,Siemens,2024,43669,19.6,3
C103,82.4447147147147,116.5,127.14474626865665,4.0603869047619,5,Compressor-103,Compressor,Plant-C,GE,2021,34630,16.2,3
C104,82.43102102102108,117.05,125.89588059701494,4.035059523809529,2,Compressor-104,Compressor,Plant-A,ABB,2023,45355,14.2,3
C105,82.75104790419157,108.93,127.11017910447771,4.151011904761905,5,Compressor-105,Compressor,Plant-B,Schneider,2019,46207,12.4,3
M101,76.2569940476191,124.02,20.994732142857153,3.196160714285714,2,Motor-101,Motor,Plant-A,ABB,2018,46109,12.8,3
M102,76.15321428571431,116.64,19.719283582089545,3.2464285714285728,1,Motor-102,Motor,Plant-B,Siemens,2020,51266,11.0,3
M103,76.74489552238803,110.52,22.09500000000001,3.156666666666665,1,Motor-103,Motor,Plant-C,GE,2016,44722,11.3,3
M104,76.48931137724546,123.34,20.80543543543543,3.2938988095238075,3,Motor-104,Motor,Plant-A,GE,2023,27208,9.100000000000001,3
M105,76.24673652694612,95.24,22.55305389221559,3.278005952380955,0,Motor-105,Motor,Plant-B,ABB,2021,27184,8.0,3


In [0]:
from pyspark.sql.functions import col

print("Total Silver Rows:", silver_sensor_df.count())

print(
    "Null Temperature:",
    silver_sensor_df.filter(col("Temperature").isNull()).count()
)

print(
    "Null Pressure:",
    silver_sensor_df.filter(col("Pressure").isNull()).count()
)

print(
    "Negative Temperature:",
    silver_sensor_df.filter(col("Temperature") < 0).count()
)

print(
    "Negative Pressure:",
    silver_sensor_df.filter(col("Pressure") < 0).count()
)

Total Silver Rows: 6720
Null Temperature: 0
Null Pressure: 0
Negative Temperature: 0
Negative Pressure: 0


In [0]:
# Simulate a new incoming IoT sensor batch

new_sensor_data_2 = [
    ("P101", "2026-08-15 01:00:00", 93.0, 112.0, 126.0, 3.3),
    ("P102", "2026-08-15 01:00:00", 89.0, 101.0, 119.0, 2.9),
    ("P104", "2026-08-15 01:00:00", 104.0, 148.0, 124.0, 7.8)
]

columns = [
    "MachineID",
    "ReadingTime",
    "Temperature",
    "Pressure",
    "Flow",
    "Vibration"
]

incoming_df = spark.createDataFrame(
    new_sensor_data_2,
    columns
)

from pyspark.sql.functions import to_timestamp

incoming_df = incoming_df.withColumn(
    "ReadingTime",
    to_timestamp("ReadingTime")
)

display(incoming_df)

MachineID,ReadingTime,Temperature,Pressure,Flow,Vibration
P101,2026-08-15T01:00:00.000Z,93.0,112.0,126.0,3.3
P102,2026-08-15T01:00:00.000Z,89.0,101.0,119.0,2.9
P104,2026-08-15T01:00:00.000Z,104.0,148.0,124.0,7.8


In [0]:
from pyspark.sql.functions import avg, max, sum, count, when, col
from delta.tables import DeltaTable

def update_gold_machine_kpi(new_silver_df):

    # Find affected machines
    affected_machines = (
        new_silver_df
        .select("MachineID")
        .distinct()
    )

    # Read complete Silver history
    silver_sensor = spark.table(
        "workspace.default.silver_sensor_readings"
    )

    # Complete history only for affected machines
    affected_sensor = silver_sensor.join(
        affected_machines,
        "MachineID",
        "inner"
    )

    # Recalculate machine sensor KPIs
    sensor_kpi = affected_sensor.groupBy("MachineID").agg(
        avg("Temperature").alias("AvgTemperature"),
        max("Temperature").alias("MaxTemperature"),
        avg("Pressure").alias("AvgPressure"),
        avg("Vibration").alias("AvgVibration"),

        sum(
            when(
                col("TemperatureStatus") == "Critical",
                1
            ).otherwise(0)
        ).alias("CriticalAlerts")
    )

    # Equipment data
    equipment = spark.table(
        "workspace.default.silver_equipment_master"
    )

    # Maintenance data
    maintenance = spark.table(
        "workspace.default.silver_maintenance_records"
    )

    maintenance_kpi = maintenance.groupBy("MachineID").agg(
        sum("Cost").alias("TotalMaintenanceCost"),
        sum("DowntimeHours").alias("TotalDowntimeHours"),
        count("MaintenanceID").alias("MaintenanceCount")
    )

    # Build updated Gold records
    updated_gold = (
        sensor_kpi
        .join(equipment, "MachineID", "left")
        .join(maintenance_kpi, "MachineID", "left")
    )

    # MERGE into Gold
    gold_table = DeltaTable.forName(
        spark,
        "workspace.default.gold_machine_kpi"
    )

    (
        gold_table.alias("target")
        .merge(
            updated_gold.alias("source"),
            "target.MachineID = source.MachineID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
from pyspark.sql.functions import to_date, avg, max, sum, when, col
from delta.tables import DeltaTable

def update_gold_daily_summary(new_silver_df):

    # Find affected machine + dates
    affected_dates = (
        new_silver_df
        .withColumn("ReadingDate", to_date("ReadingTime"))
        .select("MachineID", "ReadingDate")
        .distinct()
    )

    # Read complete Silver history
    silver_sensor = (
        spark.table(
            "workspace.default.silver_sensor_readings"
        )
        .withColumn("ReadingDate", to_date("ReadingTime"))
    )

    # Get complete data for affected machine + date
    affected_sensor = silver_sensor.join(
        affected_dates,
        ["MachineID", "ReadingDate"],
        "inner"
    )

    # Recalculate daily KPIs
    updated_daily = (
        affected_sensor
        .groupBy("MachineID", "ReadingDate")
        .agg(
            avg("Temperature").alias("AvgTemperature"),
            max("Temperature").alias("MaxTemperature"),
            avg("Pressure").alias("AvgPressure"),
            max("Vibration").alias("MaxVibration"),

            sum(
                when(
                    col("MachineStatus") == "Critical",
                    1
                ).otherwise(0)
            ).alias("CriticalAlertCount")
        )
    )

    # MERGE into daily Gold table
    gold_daily_table = DeltaTable.forName(
        spark,
        "workspace.default.gold_daily_sensor_summary"
    )

    (
        gold_daily_table.alias("target")
        .merge(
            updated_daily.alias("source"),
            """
            target.MachineID = source.MachineID
            AND target.ReadingDate = source.ReadingDate
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
# Complete incremental batch pipeline
# Safe to rerun because Bronze, Silver and Gold use MERGE/upsert logic

def process_incremental_batch(incoming_df):

    from delta.tables import DeltaTable

    # 1. MERGE incoming batch into Bronze
    bronze_table = DeltaTable.forName(
        spark,
        "workspace.default.bronze_sensor_readings"
    )

    (
        bronze_table.alias("target")
        .merge(
            incoming_df.alias("source"),
            """
            target.MachineID = source.MachineID
            AND target.ReadingTime = source.ReadingTime
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Bronze updated.")

    # 2. Apply existing Silver validation logic
    new_silver_df = clean_sensor_data(incoming_df)

    # 3. MERGE into Silver
    merge_into_silver(new_silver_df)

    print("Silver updated.")

    # 4. Recalculate affected machine KPIs
    update_gold_machine_kpi(new_silver_df)

    # 5. Recalculate affected daily KPIs
    update_gold_daily_summary(new_silver_df)

    print("Gold updated.")
    print("Incremental pipeline completed successfully.")